<a href="https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Research question

Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring based on observable search-performance signals?

### Decision it supports

The analysis supports prioritizing pages for human review. A ranked queue can help reviewers decide which pages deserve attention first and consider an appropriate action such as refresh, expansion, protection, pruning, or monitoring.

The goal is to improve review prioritization, not to automatically decide the action or guarantee that changing a page will improve future performance.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



This analysis uses the **FlyRank internship warehouse release** available through the gated Hugging Face dataset.

### Table and grain

The main source is:

- `fact_content_daily_performance`

The table is at a **daily × client × content** grain. Each row represents the search and analytics performance of one content page for one client on one day.

### Date windows

The current-window baseline uses **March 2026**:

- Start: `2026-03-01`
- End: `2026-03-31`

The March extract contains **9,841,378 daily rows**.

For temporal supervised evaluation, the analysis additionally uses consecutive monthly partitions from **January through June 2026**:

- January → February
- February → March
- March → April
- April → May
- May → June

The May → June pair is held out as the future-period test set.

### Signals used

For the page-level analysis, the daily records are aggregated by client and content page using:

- Search impressions
- Search clicks
- Search position

These are used to calculate:

- Total impressions
- Total clicks
- CTR
- Average search position

A position-bucket CTR benchmark is also calculated from the March data.

### Exclusions

The analysis does not use:

- Client names or domains
- URLs
- Private search queries
- Credentials or private information
- Raw exports
- Future-window performance metrics
- Product flags
- The starter `trend_direction` field as a baseline input

The analysis is intended to provide **public-safe, pseudonymized, decision-support analysis** rather than identify individual clients or pages publicly.

In [1]:
import os
import duckdb
from google.colab import userdata

# Get the Hugging Face token from Colab Secrets
token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token

# Create DuckDB connection
con = duckdb.connect()

# Store the token securely inside DuckDB
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("DuckDB connected successfully.")

DuckDB connected successfully.


**March 2026 data verification**

The following query verifies the row count and date range of the March 2026 extract.

In [2]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{march_path}')
""").df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


**Grain validation**

The following check verifies that the March extract contains one unique record per date, client, and content combination.

In [3]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
            AS unique_page_days
    FROM read_parquet('{march_path}')
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_days
0,9841378,9841378


**Schema inspection**

The table schema is inspected to confirm the available fields and their data types before selecting the signals used in the analysis.

In [4]:
schema_check = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

schema_check[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


**Page-level aggregation**

Daily records are aggregated by client and content page to create one March-level observation per page. Total impressions, clicks, and summed search-position values are retained for calculating CTR and impression-weighted average position.

In [5]:
page_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position
    FROM read_parquet('{march_path}')
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

page_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,sum_position
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,936.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,209.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5237.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,202.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,21612.0


**Derived page-level metrics**

CTR is calculated as clicks divided by impressions. Average search position is calculated as summed search position divided by impressions. Pages with zero impressions are assigned missing values because these metrics cannot be meaningfully measured without impressions.

In [6]:
import numpy as np

page_df["ctr"] = page_df["clicks"] / page_df["impressions"]

# No impressions means CTR cannot be measured
page_df.loc[page_df["impressions"] == 0, "ctr"] = np.nan

page_df["avg_position"] = (
    page_df["sum_position"] / page_df["impressions"]
)

page_df.loc[page_df["impressions"] == 0, "avg_position"] = np.nan

page_df.head()

,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,avg_position
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,936.0,0.000000,5.171271
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,209.0,0.021739,4.543478
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5237.0,0.001112,5.825362
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,202.0,0.000000,5.941176
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,21612.0,0.000000,6.953668


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



*Assumptions, features, target definition, baseline, temporal validation, models, and leakage checks.*

### Assumptions

The analysis uses observable search-performance signals to prioritize pages for human review.

A page with weaker performance relative to similar search-position contexts is treated as a potential review signal, not proof that the page needs a specific change or that a change will improve future performance.

The analysis is designed for **decision support**, not automatic content actions.

### Features

The temporal supervised models use signals available during the current month:

* Search impressions
* Search clicks
* Current-month CTR
* Impression-weighted average search position
* Position bucket

The position bucket groups pages into broad search-position ranges:

* Top 3
* 4–10
* 11–20
* 20+

Future-month performance metrics are used only to construct the target and are not used as model features.

### Target

For the temporal supervised models, the target is:

> **1 if the page's CTR decreases from the current month to the following month; otherwise 0.**

For example, in the May → June evaluation pair, May performance is used as the input and the change in CTR from May to June determines the target.

This target represents an observed performance decline, not a diagnosis of why the decline occurred.

### Baseline

The transparent baseline identifies pages with:

1. At least 500 current-month search impressions.
2. CTR below the benchmark for their position bucket.

The CTR gap is calculated as:

`position-benchmark CTR − page CTR`

Pages with a positive gap are candidates for review.

The baseline ranking score combines the CTR gap with log-transformed impressions so that visibility contributes to priority without allowing extremely large impression counts to dominate the ranking.

### Temporal validation

The supervised models use consecutive monthly pairs from January through June 2026:

* January → February
* February → March
* March → April
* April → May
* May → June

The first four pairs are used for model training:

* January → February
* February → March
* March → April
* April → May

The **May → June pair is held out as the future-period test set**.

This temporal split is used to avoid training on information from the evaluation period.

### Models

Two supervised ranking approaches are evaluated against the transparent baseline:

* **Logistic Regression** — uses standardized numeric features and one-hot encoded position buckets.
* **Decision Tree** — uses a shallow tree with `max_depth=3` to provide a more interpretable nonlinear comparison.

The models produce scores that are used to rank pages for review.

### Evaluation

The approaches are compared using ranking-oriented metrics on the same held-out May → June test set:

* Precision@20
* Precision@50
* Average Precision

The evaluation measures how effectively each approach places pages with observed next-month CTR declines near the top of the review queue.

### Model contributions

For Logistic Regression, transformed feature values are multiplied by the learned coefficients to estimate each feature's contribution to the model's **linear predictor (log-odds)**.

These contributions are used to create reviewer-facing reason codes.

They describe model associations with the target and should not be interpreted as causal explanations.

### Leakage checks

The analysis excludes:

* Future-window performance metrics from model features
* Product or production action flags
* The starter `trend_direction` field
* Client names, domains, URLs, and private queries
* Any feature derived from the evaluation outcome

The purpose is to ensure that the ranking uses information that would have been observable at the time the review-prioritization decision was made.


**Position buckets**

Pages are grouped into broad search-position ranges so that CTR can be compared within similar position contexts.

In [7]:
import pandas as pd
def position_bucket(position):
    if pd.isna(position):
        return "No position"
    elif position <= 3:
        return "Top 3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    else:
        return "20+"

page_df["position_bucket"] = page_df["avg_position"].apply(position_bucket)

page_df[["avg_position", "position_bucket"]].head()

,avg_position,position_bucket
0,5.171271,4-10
1,4.543478,4-10
2,5.825362,4-10
3,5.941176,4-10
4,6.953668,4-10


**Position-specific CTR benchmark**

A CTR benchmark is calculated for each position bucket using total clicks divided by total impressions within that bucket.

In [102]:
position_benchmark = (
    page_df.groupby("position_bucket")
    .agg(
        total_clicks=("clicks", "sum"),
        total_impressions=("impressions", "sum")
    )
)

position_benchmark["benchmark_ctr"] = (
    position_benchmark["total_clicks"]
    / position_benchmark["total_impressions"]
)

position_benchmark

,total_clicks,total_impressions,benchmark_ctr
position_bucket,,,
11-20,98488.0,31191659.0,0.003158
20+,81593.0,59933354.0,0.001361
4-10,481189.0,148112436.0,0.003249
No position,0.0,0.0,NaN
Top 3,160562.0,41420140.0,0.003876


**CTR opportunity signal**

The CTR gap measures how far a page's observed CTR is below the benchmark for its position bucket.

In [9]:
page_df["position_benchmark_ctr"] = (
    page_df["position_bucket"]
    .map(position_benchmark["benchmark_ctr"])
)

page_df["ctr_gap"] = (
    page_df["position_benchmark_ctr"] - page_df["ctr"]
)

page_df[
    [
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap"
    ]
].head()

,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap
0,content_d0dff76c889de68f,181.0,0.000000,5.171271,4-10,0.003249,0.003249
1,content_67741cce996cfafa,46.0,0.021739,4.543478,4-10,0.003249,-0.018490
2,content_2e6360ad20fd7107,899.0,0.001112,5.825362,4-10,0.003249,0.002136
3,content_ac8663da7484669a,34.0,0.000000,5.941176,4-10,0.003249,0.003249
4,content_65c50dfe9d87a585,3108.0,0.000000,6.953668,4-10,0.003249,0.003249


In [10]:
ranked_df = page_df[
    (page_df["impressions"] >= 500) &
    (page_df["ctr_gap"] > 0)
].copy()

print("Candidate pages:", len(ranked_df))

ranked_df[
    [
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap"
    ]
].head()

Candidate pages: 41087


,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap
2,content_2e6360ad20fd7107,899.0,0.001112,5.825362,4-10,0.003249,0.002136
4,content_65c50dfe9d87a585,3108.0,0.000000,6.953668,4-10,0.003249,0.003249
6,content_614baf2af4330bd7,772.0,0.001295,4.818653,4-10,0.003249,0.001953
10,content_cdd114d71966c437,1888.0,0.000000,10.622881,11-20,0.003158,0.003158
14,content_d8d69e789e933656,826.0,0.000000,6.504843,4-10,0.003249,0.003249


**Baseline ranking score**

The baseline ranking score combines the CTR gap with log-transformed impressions. This gives higher priority to pages that have both a larger CTR opportunity and meaningful search visibility, while reducing the influence of extremely large impression counts.


In [11]:
ranked_df["score"] = (
    np.log1p(ranked_df["impressions"]) *
    ranked_df["ctr_gap"]
)

ranked_df = ranked_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap",
        "score"
    ]
].head(20)

,rank,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap,score
0,1,content_44f34c0a90047651,212404.0,0.000113,0.665877,Top 3,0.003876,0.003763,0.046163
1,2,content_8e1334d6356668e3,134984.0,0.000007,2.693038,Top 3,0.003876,0.003869,0.045704
2,3,content_fec55986a1868d62,124075.0,0.000008,0.308426,Top 3,0.003876,0.003868,0.045371
3,4,content_9c057b66c30a3abb,83834.0,0.000012,0.116003,Top 3,0.003876,0.003864,0.043810
4,5,content_bf078007df823490,44707.0,0.000000,1.400049,Top 3,0.003876,0.003876,0.041508
5,6,content_d61fc394d10cba41,38000.0,0.000026,2.362579,Top 3,0.003876,0.003850,0.040601
6,7,content_fc67675904376267,60172.0,0.000299,2.126022,Top 3,0.003876,0.003577,0.039368
7,8,content_dc91779c3d085398,25625.0,0.000039,2.389151,Top 3,0.003876,0.003837,0.038955
8,9,content_306bc78dff1eb683,80821.0,0.000433,1.444266,Top 3,0.003876,0.003443,0.038910
9,10,content_66bf45eb0c5bb550,24259.0,0.000041,2.071726,Top 3,0.003876,0.003835,0.038722


In [103]:
ranked_df["reason_code"] = "LOW_CTR_VS_POSITION_BENCHMARK"
ranked_df["action"] = "REVIEW_CTR_OPPORTUNITY"

ranked_df["why_its_here"] = (
    "CTR is below the benchmark for the page's position range "
    "with at least 500 impressions."
)

ranked_df["what_would_make_it_wrong"] = (
    "The CTR gap may reflect query mix, page intent, or other factors "
    "not captured by this baseline."
)

ranked_df["confidence_note"] = (
    "Decision-support signal; not a prediction of future performance."
)

ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
].head(20)



,rank,content_hash_id,impressions,ctr,avg_position,ctr_gap,score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,0.000113,0.665877,0.003763,0.046163,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
1,2,content_8e1334d6356668e3,134984.0,0.000007,2.693038,0.003869,0.045704,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
2,3,content_fec55986a1868d62,124075.0,0.000008,0.308426,0.003868,0.045371,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
3,4,content_9c057b66c30a3abb,83834.0,0.000012,0.116003,0.003864,0.043810,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
4,5,content_bf078007df823490,44707.0,0.000000,1.400049,0.003876,0.041508,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
5,6,content_d61fc394d10cba41,38000.0,0.000026,2.362579,0.003850,0.040601,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
6,7,content_fc67675904376267,60172.0,0.000299,2.126022,0.003577,0.039368,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
7,8,content_dc91779c3d085398,25625.0,0.000039,2.389151,0.003837,0.038955,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
8,9,content_306bc78dff1eb683,80821.0,0.000433,1.444266,0.003443,0.038910,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY
9,10,content_66bf45eb0c5bb550,24259.0,0.000041,2.071726,0.003835,0.038722,LOW_CTR_VS_POSITION_BENCHMARK,REVIEW_CTR_OPPORTUNITY


**Baseline Top-20 review queue**

The baseline ranking is reduced to its top 20 pages to create a transparent review queue. The queue retains the ranking score, search-performance signals, position context, and baseline reason/action fields used to support human review.


In [14]:
baseline_top20 = ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
].head(20)

baseline_top20

,rank,content_hash_id,impressions,clicks,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap,score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,24.0,0.000113,0.665877,Top 3,0.003876,0.003763,0.046163,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
1,2,content_8e1334d6356668e3,134984.0,1.0,0.000007,2.693038,Top 3,0.003876,0.003869,0.045704,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
2,3,content_fec55986a1868d62,124075.0,1.0,0.000008,0.308426,Top 3,0.003876,0.003868,0.045371,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
3,4,content_9c057b66c30a3abb,83834.0,1.0,0.000012,0.116003,Top 3,0.003876,0.003864,0.043810,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
4,5,content_bf078007df823490,44707.0,0.0,0.000000,1.400049,Top 3,0.003876,0.003876,0.041508,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
5,6,content_d61fc394d10cba41,38000.0,1.0,0.000026,2.362579,Top 3,0.003876,0.003850,0.040601,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
6,7,content_fc67675904376267,60172.0,18.0,0.000299,2.126022,Top 3,0.003876,0.003577,0.039368,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
7,8,content_dc91779c3d085398,25625.0,1.0,0.000039,2.389151,Top 3,0.003876,0.003837,0.038955,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
8,9,content_306bc78dff1eb683,80821.0,35.0,0.000433,1.444266,Top 3,0.003876,0.003443,0.038910,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
9,10,content_66bf45eb0c5bb550,24259.0,1.0,0.000041,2.071726,Top 3,0.003876,0.003835,0.038722,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY


**Baseline recommendation summary**

The Top-20 baseline queue is summarized by reason code and suggested review action to show how the transparent baseline distributes its recommendations.

In [15]:
baseline_summary = (
    ranked_df.head(20)
    .groupby(["reason_code", "action"])
    .size()
    .reset_index(name="pages")
)

baseline_summary

,reason_code,action,pages
0,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY,20


In [16]:
baseline_columns = ranked_df.columns.tolist()

leakage_columns = [
    "trend_direction",
    "future_ctr",
    "future_impressions",
    "future_clicks",
    "product_flag"
]

found_leakage = [
    col for col in leakage_columns
    if col in baseline_columns
]

print("Potential leakage columns found:", found_leakage)

Potential leakage columns found: []


**Temporal page overlap**

Consecutive monthly partitions are checked for overlapping client–content pages so that current-month observations can be paired with the corresponding next-month observations for temporal evaluation.


In [45]:
month_pairs = [
    ("Jan→Feb", "2026-01", "2026-02"),
    ("Feb→Mar", "2026-02", "2026-03"),
    ("Mar→Apr", "2026-03", "2026-04"),
    ("Apr→May", "2026-04", "2026-05"),
    ("May→Jun", "2026-05", "2026-06")
]

overlap_results = []

for pair_name, current_month, next_month in month_pairs:

    current_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={current_month}/*.parquet"
    )

    next_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={next_month}/*.parquet"
    )

    result = con.sql(f"""
        WITH current_pages AS (
            SELECT DISTINCT
                client_hash_id,
                content_hash_id
            FROM read_parquet('{current_path}')
        ),
        next_pages AS (
            SELECT DISTINCT
                client_hash_id,
                content_hash_id
            FROM read_parquet('{next_path}')
        )
        SELECT COUNT(*) AS overlap_pages
        FROM current_pages c
        INNER JOIN next_pages n
            USING (client_hash_id, content_hash_id)
    """).df()

    overlap_results.append({
        "pair": pair_name,
        "overlap_pages": int(result["overlap_pages"].iloc[0])
    })

overlap_df = pd.DataFrame(overlap_results)

overlap_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pair,overlap_pages
0,Jan→Feb,261984
1,Feb→Mar,303572
2,Mar→Apr,331436
3,Apr→May,362172
4,May→Jun,389032


**Target outcome comparison**

The consecutive month-pairs are compared using impression, click, and CTR decline rates. CTR decline is used as the target because it directly represents a change in search engagement while impressions and clicks are also affected by changes in visibility.


In [94]:


target_results = []

for pair_name, current_month, next_month in month_pairs:

    current_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={current_month}/*.parquet"
    )

    next_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={next_month}/*.parquet"
    )

    pair_df = con.sql(f"""
        WITH current_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS current_impressions,
                SUM(gsc_clicks) AS current_clicks
            FROM read_parquet('{current_path}')
            GROUP BY client_hash_id, content_hash_id
        ),
        next_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS next_impressions,
                SUM(gsc_clicks) AS next_clicks
            FROM read_parquet('{next_path}')
            GROUP BY client_hash_id, content_hash_id
        )
        SELECT
            c.*,
            n.next_impressions,
            n.next_clicks,
            c.current_clicks * 1.0 / NULLIF(c.current_impressions, 0) AS current_ctr,
            n.next_clicks * 1.0 / NULLIF(n.next_impressions, 0) AS next_ctr
        FROM current_month c
        INNER JOIN next_month n
            USING (client_hash_id, content_hash_id)
        WHERE c.current_impressions >= 500
          AND n.next_impressions > 0
    """).df()

    target_results.append({
        "pair": pair_name,
        "pages": len(pair_df),
        "impression_decline_rate": (
            (pair_df["next_impressions"] < pair_df["current_impressions"]).mean() * 100
        ),
        "click_decline_rate": (
            (pair_df["next_clicks"] < pair_df["current_clicks"]).mean() * 100
        ),
        "ctr_decline_rate": (
            (pair_df["next_ctr"] < pair_df["current_ctr"]).mean() * 100
        )
    })

target_comparison_df = pd.DataFrame(target_results)

target_comparison_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pair,pages,impression_decline_rate,click_decline_rate,ctr_decline_rate
0,Jan→Feb,35729,40.930337,34.392230,45.724761
1,Feb→Mar,46152,30.780898,34.444878,52.106084
2,Mar→Apr,61846,65.826408,51.904731,55.536332
3,Apr→May,62010,71.675536,39.835510,37.144009
4,May→Jun,59874,77.935331,53.049738,44.002405


**Impression threshold analysis**

In [104]:
thresholds = [10, 50, 100, 250, 500, 1000]

pairs = [
    ("2026-01", "2026-02"),
    ("2026-02", "2026-03"),
    ("2026-03", "2026-04"),
    ("2026-04", "2026-05")

]

results = []

for current_month, next_month in pairs:

    current_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={current_month}/*.parquet"
    )

    next_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={next_month}/*.parquet"
    )

    for threshold in thresholds:

        query = f"""
        WITH current_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS current_impressions,
                SUM(gsc_clicks) AS current_clicks
            FROM read_parquet('{current_path}')
            GROUP BY client_hash_id, content_hash_id
        ),

        next_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS next_impressions,
                SUM(gsc_clicks) AS next_clicks
            FROM read_parquet('{next_path}')
            GROUP BY client_hash_id, content_hash_id
        ),

        paired AS (
            SELECT
                c.current_impressions,
                c.current_clicks,
                n.next_impressions,
                n.next_clicks,

                c.current_clicks * 1.0
                    / NULLIF(c.current_impressions, 0) AS current_ctr,

                n.next_clicks * 1.0
                    / NULLIF(n.next_impressions, 0) AS next_ctr

            FROM current_month c
            INNER JOIN next_month n
                USING (client_hash_id, content_hash_id)

            WHERE c.current_impressions > 0
              AND n.next_impressions > 0
        )

        SELECT
            COUNT(*) AS pages,

            SUM(
                CASE
                    WHEN next_ctr < current_ctr
                    THEN 1 ELSE 0
                END
            ) AS ctr_declined

        FROM paired

        WHERE current_impressions >= {threshold}
        """

        result = con.sql(query).df()

        result["pair"] = f"{current_month} → {next_month}"
        result["min_impressions"] = threshold

        result["ctr_decline_rate"] = (
            result["ctr_declined"]
            / result["pages"]
            * 100
        )

        results.append(result)

ctr_threshold_analysis = pd.concat(results, ignore_index=True)

ctr_threshold_analysis = ctr_threshold_analysis[
    [
        "pair",
        "min_impressions",
        "pages",
        "ctr_declined",
        "ctr_decline_rate"
    ]
]

ctr_threshold_analysis

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pair,min_impressions,pages,ctr_declined,ctr_decline_rate
0,2026-01 → 2026-02,10,92458,26090.0,28.218218
1,2026-01 → 2026-02,50,74029,24681.0,33.339637
2,2026-01 → 2026-02,100,63831,23484.0,36.790901
3,2026-01 → 2026-02,250,47932,20294.0,42.339147
4,2026-01 → 2026-02,500,35729,16337.0,45.724761
5,2026-01 → 2026-02,1000,24521,11618.0,47.379797
6,2026-02 → 2026-03,10,111494,35552.0,31.886918
7,2026-02 → 2026-03,50,88767,33845.0,38.127908
8,2026-02 → 2026-03,100,76702,32259.0,42.057573
9,2026-02 → 2026-03,250,59652,28546.0,47.854221


**Temporal dataset construction**

For each consecutive month-pair, current-month search-performance signals are paired with the following month's performance for the same client and content page. Pages must have at least 500 current-month impressions and nonzero next-month impressions.

The target is `1` when next-month CTR is lower than current-month CTR and `0` otherwise. Future-month metrics are used only to construct this target and are not included as model features.


In [52]:
temporal_datasets = []

for pair_name, current_month, next_month in month_pairs:

    current_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={current_month}/*.parquet"
    )

    next_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={next_month}/*.parquet"
    )

    pair_df = con.sql(f"""
        WITH current_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions,
                SUM(gsc_clicks) AS clicks,
                SUM(gsc_sum_position) AS sum_position
            FROM read_parquet('{current_path}')
            GROUP BY client_hash_id, content_hash_id
        ),

        next_month AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS next_impressions,
                SUM(gsc_clicks) AS next_clicks
            FROM read_parquet('{next_path}')
            GROUP BY client_hash_id, content_hash_id
        )

        SELECT
            c.*,
            n.next_impressions,
            n.next_clicks
        FROM current_month c
        INNER JOIN next_month n
            USING (client_hash_id, content_hash_id)

        WHERE c.impressions >= 500
          AND n.next_impressions > 0
    """).df()

    pair_df["current_ctr"] = (
        pair_df["clicks"] / pair_df["impressions"]
    )

    pair_df["avg_position"] = (
        pair_df["sum_position"] / pair_df["impressions"]
    )

    pair_df["next_ctr"] = (
        pair_df["next_clicks"] / pair_df["next_impressions"]
    )

    pair_df["target_ctr_declined"] = (
        pair_df["next_ctr"] < pair_df["current_ctr"]
    ).astype(int)

    pair_df["pair"] = pair_name

    temporal_datasets.append(pair_df)

temporal_df = pd.concat(
    temporal_datasets,
    ignore_index=True
)

temporal_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,sum_position,next_impressions,next_clicks,current_ctr,avg_position,next_ctr,target_ctr_declined,pair
0,client_e547b89c05043229,content_4baa677479d16fb5,1097.0,6.0,14961.0,1178.0,1.0,0.005469,13.638104,0.000849,1,Jan→Feb
1,client_e547b89c05043229,content_6a167adf0f285e1c,1031.0,1.0,3984.0,1529.0,8.0,0.000970,3.864210,0.005232,0,Jan→Feb
2,client_e547b89c05043229,content_b2c8677f822e152a,4603.0,14.0,24340.0,5197.0,23.0,0.003041,5.287856,0.004426,0,Jan→Feb
3,client_e547b89c05043229,content_c4a9f203b6708ddf,2031.0,2.0,10315.0,2081.0,5.0,0.000985,5.078779,0.002403,0,Jan→Feb
4,client_e547b89c05043229,content_20b42dcd3e0fd089,1327.0,7.0,3460.0,3311.0,10.0,0.005275,2.607385,0.003020,1,Jan→Feb


**Target distribution**

The target distribution is checked to identify whether CTR decline is severely imbalanced across the temporally paired observations.

In [53]:
temporal_df["target_ctr_declined"].value_counts(normalize=True) * 100

,proportion
target_ctr_declined,
0,53.273396
1,46.726604


**Temporal position buckets**

The current-month average search position is grouped into the same broad position buckets used by the baseline so that the feature representation remains consistent across the ranking approaches.


In [105]:
temporal_df["position_bucket"] = (
    temporal_df["avg_position"].apply(position_bucket)
)

temporal_df[
    ["avg_position", "position_bucket"]
].head(10)

,avg_position,position_bucket
0,13.638104,11-20
1,3.864210,4-10
2,5.287856,4-10
3,5.078779,4-10
4,2.607385,Top 3
5,8.556894,4-10
6,13.110065,11-20
7,5.193583,4-10
8,11.625840,11-20
9,10.983407,11-20


**Feature preparation**

The supervised models use only information available in the current month:

- `impressions`
- `clicks`
- `current_ctr`
- `avg_position`
- `position_bucket`

The target, `target_ctr_declined`, indicates whether CTR declined in the following month. Future-month performance is used only to construct this target and is not included in `X`.

In [106]:
feature_cols = [
    "impressions",
    "clicks",
    "current_ctr",
    "avg_position",
    "position_bucket"
]



In [59]:
temporal_df["pair"].value_counts().sort_index()

,count
pair,
Apr→May,62010
Feb→Mar,46152
Jan→Feb,35729
Mar→Apr,61846
May→Jun,59874


**Temporal train/test split**

The supervised models are trained on the earlier month-pairs from January–February through April–May 2026.

The May–June 2026 pair is held out as the future-period test set. This temporal split ensures that model evaluation uses a period that was not used for training.

In [60]:
train_pairs = [
    "Jan→Feb",
    "Feb→Mar",
    "Mar→Apr",
    "Apr→May"
]

test_pair = "May→Jun"

train_df = temporal_df[
    temporal_df["pair"].isin(train_pairs)
].copy()

test_df = temporal_df[
    temporal_df["pair"] == test_pair
].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("\nTrain pairs:")
print(train_df["pair"].value_counts().sort_index())
print("\nTest pair:")
print(test_df["pair"].value_counts())

Train rows: 205737
Test rows: 59874

Train pairs:
pair
Apr→May    62010
Feb→Mar    46152
Jan→Feb    35729
Mar→Apr    61846
Name: count, dtype: int64

Test pair:
pair
May→Jun    59874
Name: count, dtype: int64


In [61]:
feature_cols = [
    "impressions",
    "clicks",
    "current_ctr",
    "avg_position",
    "position_bucket"
]

X_train = train_df[feature_cols].copy()
y_train = train_df["target_ctr_declined"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target_ctr_declined"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (205737, 5)
y_train: (205737,)
X_test: (59874, 5)
y_test: (59874,)


**Logistic Regression**

Logistic Regression is used as the first supervised ranking model. Numerical features are standardized and the position bucket is one-hot encoded within a pipeline.

In [107]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "impressions",
    "clicks",
    "current_ctr",
    "avg_position"
]

categorical_features = [
    "position_bucket"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['impressions', 'clicks',
                                                   'current_ctr',
                                                   'avg_position']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['position_bucket'])])),
                ('model', LogisticRegression(max_iter=1000))])

 **Logistic Regression predictions**

The trained Logistic Regression model is applied to the held-out May–June 2026 test set. The predicted probability of CTR decline is stored as `score` and used to rank pages for review.

In [65]:
test_proba = logistic_model.predict_proba(X_test)[:, 1]

test_predictions = test_df[
    ["client_hash_id", "content_hash_id", "pair", "target_ctr_declined"]
].copy()

test_predictions["score"] = test_proba

test_predictions.head()

,client_hash_id,content_hash_id,pair,target_ctr_declined,score
205737,client_e547b89c05043229,content_2e296120acb03e93,May→Jun,1,0.265907
205738,client_e547b89c05043229,content_516b7c0e8eec0cef,May→Jun,0,0.233549
205739,client_e547b89c05043229,content_38b6c1a9aa29f801,May→Jun,1,0.285254
205740,client_e547b89c05043229,content_2ffd36f2a70be7e3,May→Jun,1,0.330378
205741,client_e547b89c05043229,content_4724385fe790d24a,May→Jun,1,0.533100


In [66]:
ranked_test = test_predictions.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_test["rank"] = ranked_test.index + 1

ranked_test.head(20)

,client_hash_id,content_hash_id,pair,target_ctr_declined,score,rank
0,client_20259bd6705d81d4,content_95dff98b3e9e533d,May→Jun,1,1.0,1
1,client_b77d0d5f08f05e64,content_fd99aed0910e521d,May→Jun,1,1.0,2
2,client_20259bd6705d81d4,content_421766adc84fcd7f,May→Jun,1,1.0,3
3,client_20259bd6705d81d4,content_e31548c47c905839,May→Jun,1,1.0,4
4,client_20259bd6705d81d4,content_7811a2ed858d5816,May→Jun,1,1.0,5
5,client_20259bd6705d81d4,content_dc5e5285b1691ea7,May→Jun,1,1.0,6
6,client_8ddc46da5414ffd8,content_d46321b2dce9da21,May→Jun,0,1.0,7
7,client_20259bd6705d81d4,content_33f041db9383a3af,May→Jun,1,1.0,8
8,client_20259bd6705d81d4,content_d15252a62541754e,May→Jun,1,1.0,9
9,client_20259bd6705d81d4,content_fa3db31b4aeeda73,May→Jun,1,1.0,10


In [69]:
def precision_at_k(df, k):
    top_k = df.head(k)
    return top_k["target_ctr_declined"].mean()

p20 = precision_at_k(ranked_test, 20)
p50 = precision_at_k(ranked_test, 50)

print(f"Precision@20: {p20:.3f}")
print(f"Precision@50: {p50:.3f}")

Precision@20: 0.800
Precision@50: 0.740


**Baseline evaluation on the held-out period**

The transparent baseline is reconstructed using only training-period CTR benchmarks and then applied to the held-out May–June 2026 test set. This keeps the baseline evaluation temporally consistent with the supervised models.

In [108]:
test_baseline = test_df.copy()

train_benchmark = (
    train_df.groupby("position_bucket")
    .agg(
        benchmark_clicks=("clicks", "sum"),
        benchmark_impressions=("impressions", "sum")
    )
)

train_benchmark["benchmark_ctr"] = (
    train_benchmark["benchmark_clicks"]
    / train_benchmark["benchmark_impressions"]
)

test_baseline = test_baseline.merge(
    train_benchmark[["benchmark_ctr"]],
    left_on="position_bucket",
    right_index=True,
    how="left"
)

test_baseline["ctr_gap"] = (
    test_baseline["benchmark_ctr"]
    - test_baseline["current_ctr"]
)

test_baseline["baseline_score"] = (
    np.log1p(test_baseline["impressions"])
    * test_baseline["ctr_gap"]
)

test_baseline = test_baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

test_baseline["rank"] = (
    test_baseline.index + 1
)

test_baseline.head(20)

,client_hash_id,content_hash_id,impressions,clicks,sum_position,next_impressions,next_clicks,current_ctr,avg_position,next_ctr,target_ctr_declined,pair,position_bucket,benchmark_ctr,ctr_gap,baseline_score,rank
0,client_157ffe4d4a595515,content_9648c4d1595a0794,219982.0,52.0,428133.0,6001.0,12.0,0.000236,1.946218,0.002000,0,May→Jun,Top3,0.003955,0.003718,0.045742,1
1,client_23a62021009f63c4,content_bddfdd871aa09fbe,112353.0,4.0,131913.0,49074.0,22.0,0.000036,1.174094,0.000448,0,May→Jun,Top3,0.003955,0.003919,0.045578,2
2,client_8ddc46da5414ffd8,content_576d3cd50e6243c8,39579.0,5.0,80281.0,20976.0,2.0,0.000126,2.028374,0.000095,1,May→Jun,Top3,0.003955,0.003828,0.040529,3
3,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,129493.0,72.0,288404.0,145022.0,46.0,0.000556,2.227178,0.000317,1,May→Jun,Top3,0.003955,0.003399,0.040009,4
4,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,189298.0,2.0,1908139.0,6777.0,4.0,0.000011,10.080080,0.000590,0,May→Jun,11-20,0.003258,0.003248,0.039465,5
5,client_73cda7b4e4f265ea,content_33d31496fca9665e,235817.0,45.0,1363346.0,169233.0,23.0,0.000191,5.781373,0.000136,1,May→Jun,4-10,0.003263,0.003072,0.038005,6
6,client_23a62021009f63c4,content_dd159d26bd77a5ad,14468.0,0.0,33903.0,35141.0,5.0,0.000000,2.343309,0.000142,0,May→Jun,Top3,0.003955,0.003955,0.037886,7
7,client_23a62021009f63c4,content_65c75874a23fca87,191566.0,32.0,1666112.0,22969.0,12.0,0.000167,8.697326,0.000522,0,May→Jun,4-10,0.003263,0.003096,0.037656,8
8,client_08a6a72ff48e62c0,content_13ef916471e84dd8,14584.0,1.0,31474.0,19.0,0.0,0.000069,2.158118,0.000000,1,May→Jun,Top3,0.003955,0.003886,0.037260,9
9,client_23a62021009f63c4,content_6982fdcd6a6b28f8,19659.0,4.0,52685.0,3565.0,3.0,0.000203,2.679943,0.000842,0,May→Jun,Top3,0.003955,0.003751,0.037087,10


In [109]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        ))
    ]
)

tree_model.fit(X_train, y_train)

tree_proba = tree_model.predict_proba(X_test)[:, 1]

tree_predictions = test_df[
    ["client_hash_id", "content_hash_id", "pair", "target_ctr_declined"]
].copy()

tree_predictions["score"] = tree_proba

ranked_tree = tree_predictions.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_tree["rank"] = ranked_tree.index + 1



 **Logistic Regression feature contributions**

The Logistic Regression coefficients are inspected to identify which observed signals contribute most strongly to the model score. These contributions are used as associative reason codes for reviewer interpretation and are not treated as causal explanations.

In [75]:
feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logistic_model.named_steps[
    "model"
].coef_[0]

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df.sort_values(
    "abs_coefficient",
    ascending=False
)

,feature,coefficient,abs_coefficient
2,numeric__current_ctr,1.070155,1.070155
1,numeric__clicks,-0.228144,0.228144
7,categorical__position_bucket_Top3,-0.201560,0.201560
0,numeric__impressions,0.175667,0.175667
5,categorical__position_bucket_20+,0.173954,0.173954
3,numeric__avg_position,-0.146044,0.146044
4,categorical__position_bucket_11-20,0.062503,0.062503
6,categorical__position_bucket_4-10,-0.049196,0.049196


**Model contribution analysis**

For each held-out test page, the transformed feature values are multiplied by the Logistic Regression coefficients to estimate each feature's contribution to the model's linear predictor (log-odds).

These contributions are used to create reviewer-facing reason codes. They describe model associations, not causal explanations.

In [78]:
# Transform the test features exactly as Logistic Regression sees them
X_test_transformed = logistic_model.named_steps[
    "preprocessor"
].transform(X_test)

# Get the model coefficients
coefficients = logistic_model.named_steps[
    "model"
].coef_[0]

# Calculate contribution of each feature
contribution_values = X_test_transformed * coefficients

# Convert to DataFrame
contribution_df = pd.DataFrame(
    contribution_values,
    columns=logistic_model.named_steps[
        "preprocessor"
    ].get_feature_names_out()
)

contribution_df.head()

,numeric__impressions,numeric__clicks,numeric__current_ctr,numeric__avg_position,categorical__position_bucket_11-20,categorical__position_bucket_20+,categorical__position_bucket_4-10,categorical__position_bucket_Top3
0,-0.023238,0.060001,-0.751111,-0.460753,0.000000,0.173954,-0.0,-0.0
1,-0.069632,0.065125,-0.848645,-0.494842,0.000000,0.173954,-0.0,-0.0
2,0.036870,0.049752,-0.697295,-0.467484,0.000000,0.173954,-0.0,-0.0
3,-0.053716,0.060001,-0.663494,-0.208879,0.000000,0.173954,-0.0,-0.0
4,-0.054774,0.039503,0.106916,-0.007213,0.062503,0.000000,-0.0,-0.0


In [79]:
# Keep the original test-page information
ranked_with_contributions = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "pair",
        "impressions",
        "clicks",
        "current_ctr",
        "avg_position",
        "position_bucket",
        "target_ctr_declined"
    ]
].copy()

# Add the Logistic Regression score
ranked_with_contributions["score"] = test_proba

# Add model contributions
ranked_with_contributions = pd.concat(
    [
        ranked_with_contributions.reset_index(drop=True),
        contribution_df.reset_index(drop=True)
    ],
    axis=1
)

# Rank by model score
ranked_with_contributions = ranked_with_contributions.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_with_contributions["rank"] = (
    ranked_with_contributions.index + 1
)

ranked_with_contributions.head(20)

,client_hash_id,content_hash_id,pair,impressions,clicks,current_ctr,avg_position,position_bucket,target_ctr_declined,score,numeric__impressions,numeric__clicks,numeric__current_ctr,numeric__avg_position,categorical__position_bucket_11-20,categorical__position_bucket_20+,categorical__position_bucket_4-10,categorical__position_bucket_Top3,rank
0,client_20259bd6705d81d4,content_95dff98b3e9e533d,May→Jun,1476.0,184.0,0.124661,4.405827,4-10,1,1.0,-0.056404,-0.877750,36.150473,0.087362,0.000000,0.000000,-0.049196,-0.00000,1
1,client_b77d0d5f08f05e64,content_fd99aed0910e521d,May→Jun,5409.0,571.0,0.105565,1.946755,Top3,1,1.0,0.026837,-2.860864,30.482699,0.121764,0.000000,0.000000,-0.000000,-0.20156,2
2,client_20259bd6705d81d4,content_421766adc84fcd7f,May→Jun,761.0,66.0,0.086728,4.643890,4-10,1,1.0,-0.071537,-0.273080,24.891986,0.084031,0.000000,0.000000,-0.049196,-0.00000,3
3,client_20259bd6705d81d4,content_e31548c47c905839,May→Jun,1698.0,147.0,0.086572,5.262662,4-10,1,1.0,-0.051705,-0.688150,24.845818,0.075375,0.000000,0.000000,-0.049196,-0.00000,4
4,client_20259bd6705d81d4,content_7811a2ed858d5816,May→Jun,606.0,49.0,0.080858,5.584158,4-10,1,1.0,-0.074817,-0.185967,23.149814,0.070877,0.000000,0.000000,-0.049196,-0.00000,5
5,client_20259bd6705d81d4,content_dc5e5285b1691ea7,May→Jun,521.0,42.0,0.080614,7.886756,4-10,1,1.0,-0.076616,-0.150097,23.077431,0.038665,0.000000,0.000000,-0.049196,-0.00000,6
6,client_8ddc46da5414ffd8,content_d46321b2dce9da21,May→Jun,756.0,52.0,0.068783,4.503968,4-10,0,1.0,-0.071642,-0.201340,19.565982,0.085989,0.000000,0.000000,-0.049196,-0.00000,7
7,client_20259bd6705d81d4,content_33f041db9383a3af,May→Jun,607.0,41.0,0.067545,6.214168,4-10,1,1.0,-0.074796,-0.144972,19.198617,0.062064,0.000000,0.000000,-0.049196,-0.00000,8
8,client_20259bd6705d81d4,content_d15252a62541754e,May→Jun,1011.0,67.0,0.066271,10.229476,11-20,1,1.0,-0.066245,-0.278205,18.820413,0.005891,0.062503,0.000000,-0.000000,-0.00000,9
9,client_20259bd6705d81d4,content_fa3db31b4aeeda73,May→Jun,637.0,42.0,0.065934,7.998430,4-10,1,1.0,-0.074161,-0.150097,18.720406,0.037102,0.000000,0.000000,-0.049196,-0.00000,10


In [96]:
# Contribution columns from the Logistic Regression
contribution_cols = [
    col for col in ranked_with_contributions.columns
    if col.startswith("numeric__")
    or col.startswith("categorical__position_bucket_")
]

# Find the strongest positive contribution for each page
def get_model_reason(row):
    positive_contributions = row[contribution_cols][
        row[contribution_cols] > 0
    ]

    if positive_contributions.empty:
        return "NO_POSITIVE_FEATURE_CONTRIBUTION"

    strongest_feature = positive_contributions.idxmax()

    reason_map = {
        "numeric__impressions":
            "MODEL_SIGNAL_IMPRESSIONS",
        "numeric__clicks":
            "MODEL_SIGNAL_CLICKS",
        "numeric__current_ctr":
            "MODEL_SIGNAL_CURRENT_CTR",
        "numeric__avg_position":
            "MODEL_SIGNAL_AVG_POSITION",
        "categorical__position_bucket_11-20":
            "MODEL_SIGNAL_POSITION_11_20",
        "categorical__position_bucket_20+":
            "MODEL_SIGNAL_POSITION_20_PLUS",
        "categorical__position_bucket_4-10":
            "MODEL_SIGNAL_POSITION_4_10",
        "categorical__position_bucket_Top 3":
            "MODEL_SIGNAL_POSITION_TOP3"
    }

    return reason_map[strongest_feature]


ranked_with_contributions["reason_code"] = (
    ranked_with_contributions.apply(
        get_model_reason,
        axis=1
    )
)

ranked_with_contributions[
    [
        "rank",
        "score",
        "current_ctr",
        "avg_position",
        "reason_code"
    ]
].head(20)

,rank,score,current_ctr,avg_position,reason_code
0,1,1.0,0.124661,4.405827,MODEL_SIGNAL_CURRENT_CTR
1,2,1.0,0.105565,1.946755,MODEL_SIGNAL_CURRENT_CTR
2,3,1.0,0.086728,4.643890,MODEL_SIGNAL_CURRENT_CTR
3,4,1.0,0.086572,5.262662,MODEL_SIGNAL_CURRENT_CTR
4,5,1.0,0.080858,5.584158,MODEL_SIGNAL_CURRENT_CTR
5,6,1.0,0.080614,7.886756,MODEL_SIGNAL_CURRENT_CTR
6,7,1.0,0.068783,4.503968,MODEL_SIGNAL_CURRENT_CTR
7,8,1.0,0.067545,6.214168,MODEL_SIGNAL_CURRENT_CTR
8,9,1.0,0.066271,10.229476,MODEL_SIGNAL_CURRENT_CTR
9,10,1.0,0.065934,7.998430,MODEL_SIGNAL_CURRENT_CTR


In [99]:
def get_second_reason(row):
    positive_contributions = row[contribution_cols][
        row[contribution_cols] > 0
    ].sort_values(ascending=False)

    if len(positive_contributions) < 2:
        return "NONE"

    second_feature = positive_contributions.index[1]

    reason_map = {
        "numeric__impressions":
            "MODEL_SIGNAL_IMPRESSIONS",
        "numeric__clicks":
            "MODEL_SIGNAL_CLICKS",
        "numeric__current_ctr":
            "MODEL_SIGNAL_CURRENT_CTR",
        "numeric__avg_position":
            "MODEL_SIGNAL_AVG_POSITION",
        "categorical__position_bucket_11-20":
            "MODEL_SIGNAL_POSITION_11_20",
        "categorical__position_bucket_20+":
            "MODEL_SIGNAL_POSITION_20_PLUS",
        "categorical__position_bucket_4-10":
            "MODEL_SIGNAL_POSITION_4_10",
        "categorical__position_bucket_Top 3":
            "MODEL_SIGNAL_POSITION_TOP3"
    }

    return reason_map[second_feature]


ranked_with_contributions["reason_code_2"] = (
    ranked_with_contributions.apply(
        get_second_reason,
        axis=1
    )
)

ranked_with_contributions[
    [
        "rank",
        "score",
        "current_ctr",
        "avg_position",
        "reason_code",
        "reason_code_2"
    ]
].head(20)

,rank,score,current_ctr,avg_position,reason_code,reason_code_2
0,1,1.0,0.124661,4.405827,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
1,2,1.0,0.105565,1.946755,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
2,3,1.0,0.086728,4.643890,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
3,4,1.0,0.086572,5.262662,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
4,5,1.0,0.080858,5.584158,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
5,6,1.0,0.080614,7.886756,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
6,7,1.0,0.068783,4.503968,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
7,8,1.0,0.067545,6.214168,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION
8,9,1.0,0.066271,10.229476,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_POSITION_11_20
9,10,1.0,0.065934,7.998430,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION


In [100]:
final_queue = ranked_with_contributions.copy()

# Find the two strongest positive model contributions
primary_contributions = []
secondary_contributions = []

for _, row in final_queue.iterrows():

    positive = row[contribution_cols][
        row[contribution_cols] > 0
    ].sort_values(ascending=False)

    primary_contributions.append(
        positive.iloc[0] if len(positive) >= 1 else np.nan
    )

    secondary_contributions.append(
        positive.iloc[1] if len(positive) >= 2 else np.nan
    )

final_queue["primary_contribution"] = primary_contributions
final_queue["secondary_contribution"] = secondary_contributions

# Keep only the final reviewer-facing columns
final_queue = final_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "impressions",
        "clicks",
        "current_ctr",
        "avg_position",
        "position_bucket",
        "reason_code",
        "primary_contribution",
        "reason_code_2",
        "secondary_contribution",
        "target_ctr_declined"
    ]
]

final_queue.head(20)

,rank,client_hash_id,content_hash_id,score,impressions,clicks,current_ctr,avg_position,position_bucket,reason_code,primary_contribution,reason_code_2,secondary_contribution,target_ctr_declined
0,1,client_20259bd6705d81d4,content_95dff98b3e9e533d,1.0,1476.0,184.0,0.124661,4.405827,4-10,MODEL_SIGNAL_CURRENT_CTR,36.150473,MODEL_SIGNAL_AVG_POSITION,0.087362,1
1,2,client_b77d0d5f08f05e64,content_fd99aed0910e521d,1.0,5409.0,571.0,0.105565,1.946755,Top3,MODEL_SIGNAL_CURRENT_CTR,30.482699,MODEL_SIGNAL_AVG_POSITION,0.121764,1
2,3,client_20259bd6705d81d4,content_421766adc84fcd7f,1.0,761.0,66.0,0.086728,4.643890,4-10,MODEL_SIGNAL_CURRENT_CTR,24.891986,MODEL_SIGNAL_AVG_POSITION,0.084031,1
3,4,client_20259bd6705d81d4,content_e31548c47c905839,1.0,1698.0,147.0,0.086572,5.262662,4-10,MODEL_SIGNAL_CURRENT_CTR,24.845818,MODEL_SIGNAL_AVG_POSITION,0.075375,1
4,5,client_20259bd6705d81d4,content_7811a2ed858d5816,1.0,606.0,49.0,0.080858,5.584158,4-10,MODEL_SIGNAL_CURRENT_CTR,23.149814,MODEL_SIGNAL_AVG_POSITION,0.070877,1
5,6,client_20259bd6705d81d4,content_dc5e5285b1691ea7,1.0,521.0,42.0,0.080614,7.886756,4-10,MODEL_SIGNAL_CURRENT_CTR,23.077431,MODEL_SIGNAL_AVG_POSITION,0.038665,1
6,7,client_8ddc46da5414ffd8,content_d46321b2dce9da21,1.0,756.0,52.0,0.068783,4.503968,4-10,MODEL_SIGNAL_CURRENT_CTR,19.565982,MODEL_SIGNAL_AVG_POSITION,0.085989,0
7,8,client_20259bd6705d81d4,content_33f041db9383a3af,1.0,607.0,41.0,0.067545,6.214168,4-10,MODEL_SIGNAL_CURRENT_CTR,19.198617,MODEL_SIGNAL_AVG_POSITION,0.062064,1
8,9,client_20259bd6705d81d4,content_d15252a62541754e,1.0,1011.0,67.0,0.066271,10.229476,11-20,MODEL_SIGNAL_CURRENT_CTR,18.820413,MODEL_SIGNAL_POSITION_11_20,0.062503,1
9,10,client_20259bd6705d81d4,content_fa3db31b4aeeda73,1.0,637.0,42.0,0.065934,7.998430,4-10,MODEL_SIGNAL_CURRENT_CTR,18.720406,MODEL_SIGNAL_AVG_POSITION,0.037102,1


In [84]:
k_values = [10, 20, 30, 50, 100]

evaluation_rows = []

for k in k_values:
    lr_precision = (
        ranked_test.head(k)["target_ctr_declined"].mean()
    )

    baseline_precision = (
        test_baseline.head(k)["target_ctr_declined"].mean()
    )

    evaluation_rows.append({
        "K": k,
        "Logistic_Regression_Precision": lr_precision,
        "Baseline_Precision": baseline_precision,
        "LR_lift": lr_precision - baseline_precision
    })

top_k_comparison = pd.DataFrame(evaluation_rows)

top_k_comparison

,K,Logistic_Regression_Precision,Baseline_Precision,LR_lift
0,10,0.90,0.30,0.60
1,20,0.80,0.30,0.50
2,30,0.80,0.30,0.50
3,50,0.74,0.26,0.48
4,100,0.74,0.30,0.44


**Reviewer category**

The final queue assigns each page a reviewer-oriented category using current CTR and average search position.

These categories are rule-based review prompts. They are not model predictions and do not automatically determine the final SEO action.

In [86]:
def assign_review_category(row):
    if row["avg_position"] <= 10 and row["current_ctr"] < 0.001:
        return "CTR_OPPORTUNITY_REVIEW"

    elif row["avg_position"] > 20:
        return "VISIBILITY_REVIEW"

    elif row["avg_position"] > 10 and row["current_ctr"] < 0.002:
        return "MID_RANKING_REVIEW"

    else:
        return "PERFORMANCE_REVIEW"


final_queue["review_category"] = (
    final_queue.apply(
        assign_review_category,
        axis=1
    )
)

final_queue.head(20)

,rank,client_hash_id,content_hash_id,score,impressions,clicks,current_ctr,avg_position,position_bucket,reason_code,primary_contribution,reason_code_2,secondary_contribution,target_ctr_declined,review_category
0,1,client_20259bd6705d81d4,content_95dff98b3e9e533d,1.0,1476.0,184.0,0.124661,4.405827,4-10,MODEL_SIGNAL_CURRENT_CTR,36.150473,MODEL_SIGNAL_AVG_POSITION,0.087362,1,PERFORMANCE_REVIEW
1,2,client_b77d0d5f08f05e64,content_fd99aed0910e521d,1.0,5409.0,571.0,0.105565,1.946755,Top3,MODEL_SIGNAL_CURRENT_CTR,30.482699,MODEL_SIGNAL_AVG_POSITION,0.121764,1,PERFORMANCE_REVIEW
2,3,client_20259bd6705d81d4,content_421766adc84fcd7f,1.0,761.0,66.0,0.086728,4.643890,4-10,MODEL_SIGNAL_CURRENT_CTR,24.891986,MODEL_SIGNAL_AVG_POSITION,0.084031,1,PERFORMANCE_REVIEW
3,4,client_20259bd6705d81d4,content_e31548c47c905839,1.0,1698.0,147.0,0.086572,5.262662,4-10,MODEL_SIGNAL_CURRENT_CTR,24.845818,MODEL_SIGNAL_AVG_POSITION,0.075375,1,PERFORMANCE_REVIEW
4,5,client_20259bd6705d81d4,content_7811a2ed858d5816,1.0,606.0,49.0,0.080858,5.584158,4-10,MODEL_SIGNAL_CURRENT_CTR,23.149814,MODEL_SIGNAL_AVG_POSITION,0.070877,1,PERFORMANCE_REVIEW
5,6,client_20259bd6705d81d4,content_dc5e5285b1691ea7,1.0,521.0,42.0,0.080614,7.886756,4-10,MODEL_SIGNAL_CURRENT_CTR,23.077431,MODEL_SIGNAL_AVG_POSITION,0.038665,1,PERFORMANCE_REVIEW
6,7,client_8ddc46da5414ffd8,content_d46321b2dce9da21,1.0,756.0,52.0,0.068783,4.503968,4-10,MODEL_SIGNAL_CURRENT_CTR,19.565982,MODEL_SIGNAL_AVG_POSITION,0.085989,0,PERFORMANCE_REVIEW
7,8,client_20259bd6705d81d4,content_33f041db9383a3af,1.0,607.0,41.0,0.067545,6.214168,4-10,MODEL_SIGNAL_CURRENT_CTR,19.198617,MODEL_SIGNAL_AVG_POSITION,0.062064,1,PERFORMANCE_REVIEW
8,9,client_20259bd6705d81d4,content_d15252a62541754e,1.0,1011.0,67.0,0.066271,10.229476,11-20,MODEL_SIGNAL_CURRENT_CTR,18.820413,MODEL_SIGNAL_POSITION_11_20,0.062503,1,PERFORMANCE_REVIEW
9,10,client_20259bd6705d81d4,content_fa3db31b4aeeda73,1.0,637.0,42.0,0.065934,7.998430,4-10,MODEL_SIGNAL_CURRENT_CTR,18.720406,MODEL_SIGNAL_AVG_POSITION,0.037102,1,PERFORMANCE_REVIEW


In [87]:
reviewer_queue = final_queue[
    [
        "rank",
        "content_hash_id",
        "score",
        "impressions",
        "current_ctr",
        "avg_position",
        "position_bucket",
        "reason_code",
        "reason_code_2",
        "review_category"
    ]
].copy()

reviewer_queue.head(20)

,rank,content_hash_id,score,impressions,current_ctr,avg_position,position_bucket,reason_code,reason_code_2,review_category
0,1,content_95dff98b3e9e533d,1.0,1476.0,0.124661,4.405827,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
1,2,content_fd99aed0910e521d,1.0,5409.0,0.105565,1.946755,Top3,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
2,3,content_421766adc84fcd7f,1.0,761.0,0.086728,4.643890,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
3,4,content_e31548c47c905839,1.0,1698.0,0.086572,5.262662,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
4,5,content_7811a2ed858d5816,1.0,606.0,0.080858,5.584158,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
5,6,content_dc5e5285b1691ea7,1.0,521.0,0.080614,7.886756,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
6,7,content_d46321b2dce9da21,1.0,756.0,0.068783,4.503968,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
7,8,content_33f041db9383a3af,1.0,607.0,0.067545,6.214168,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
8,9,content_d15252a62541754e,1.0,1011.0,0.066271,10.229476,11-20,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_POSITION_11_20,PERFORMANCE_REVIEW
9,10,content_fa3db31b4aeeda73,1.0,637.0,0.065934,7.998430,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW


**Model comparison**

The transparent baseline, Logistic Regression, and Decision Tree are evaluated on the same held-out May–June 2026 test period.

The comparison uses Precision@20, Precision@50, and Average Precision. All metrics are calculated directly from the held-out predictions to keep the evaluation reproducible.

In [101]:
model_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Precision@20": [
        test_baseline.head(20)["target_ctr_declined"].mean(),
        ranked_test.head(20)["target_ctr_declined"].mean(),
        ranked_tree.head(20)["target_ctr_declined"].mean()
    ],
    "Precision@50": [
        test_baseline.head(50)["target_ctr_declined"].mean(),
        ranked_test.head(50)["target_ctr_declined"].mean(),
        ranked_tree.head(50)["target_ctr_declined"].mean()
    ],
    "Average Precision": [
        average_precision_score(
            test_baseline["target_ctr_declined"],
            test_baseline["baseline_score"]
        ),
        average_precision_score(
            ranked_test["target_ctr_declined"],
            ranked_test["score"]
        ),
        average_precision_score(
            ranked_tree["target_ctr_declined"],
            ranked_tree["score"]
        )
    ]
})

model_comparison

,Model,Precision@20,Precision@50,Average Precision
0,Baseline,0.3,0.26,0.330741
1,Logistic Regression,0.8,0.74,0.647204
2,Decision Tree,0.6,0.62,0.604717


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*



The models were evaluated on a temporally held-out May→June 2026 period. The target was whether a page's CTR declined from the current month to the following month.

The transparent baseline achieved Precision@20 of 0.300 and Precision@50 of 0.260, with Average Precision of 0.331.

Logistic Regression achieved Precision@20 of 0.800 and Precision@50 of 0.740, with Average Precision of 0.647.

The Decision Tree achieved Precision@20 of 0.600 and Precision@50 of 0.620, with Average Precision of 0.605.

For the held-out May→June evaluation period, Logistic Regression produced the highest values among the tested approaches across the reported ranking metrics. At K=20, 16 of the top 20 Logistic Regression-ranked pages experienced the target CTR decline, compared with 6 of the top 20 pages selected by the baseline.

The results support using the Logistic Regression ranking as the current review-prioritization model for this evaluation period. They do not establish that the model will perform equally well on future periods or that reviewing or changing a page will cause its future performance to improve.

The final reviewer queue therefore uses the Logistic Regression score to prioritize pages and model-derived contribution codes to explain the strongest signals associated with each ranking.


In [92]:
from sklearn.metrics import average_precision_score

model_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Precision@20": [
        test_baseline.head(20)["target_ctr_declined"].mean(),
        ranked_test.head(20)["target_ctr_declined"].mean(),
        ranked_tree.head(20)["target_ctr_declined"].mean()
    ],
    "Precision@50": [
        test_baseline.head(50)["target_ctr_declined"].mean(),
        ranked_test.head(50)["target_ctr_declined"].mean(),
        ranked_tree.head(50)["target_ctr_declined"].mean()
    ],
    "Average Precision": [
        average_precision_score(
            test_baseline["target_ctr_declined"],
            test_baseline["baseline_score"]
        ),
        average_precision_score(
            ranked_test["target_ctr_declined"],
            ranked_test["score"]
        ),
        average_precision_score(
            ranked_tree["target_ctr_declined"],
            ranked_tree["score"]
        )
    ]
})

model_comparison

,Model,Precision@20,Precision@50,Average Precision
0,Baseline,0.3,0.26,0.330741
1,Logistic Regression,0.8,0.74,0.647204
2,Decision Tree,0.6,0.62,0.604717


In [90]:
k_values = [10, 20, 30, 50, 100]

evaluation_rows = []

for k in k_values:
    lr_precision = (
        ranked_test.head(k)["target_ctr_declined"].mean()
    )

    baseline_precision = (
        test_baseline.head(k)["target_ctr_declined"].mean()
    )

    evaluation_rows.append({
        "K": k,
        "Logistic_Regression_Precision": lr_precision,
        "Baseline_Precision": baseline_precision,
        "LR_lift": lr_precision - baseline_precision
    })

top_k_comparison = pd.DataFrame(evaluation_rows)

top_k_comparison

,K,Logistic_Regression_Precision,Baseline_Precision,LR_lift
0,10,0.90,0.30,0.60
1,20,0.80,0.30,0.50
2,30,0.80,0.30,0.50
3,50,0.74,0.26,0.48
4,100,0.74,0.30,0.44


In [91]:
reviewer_queue = final_queue[
    [
        "rank",
        "content_hash_id",
        "score",
        "impressions",
        "current_ctr",
        "avg_position",
        "position_bucket",
        "reason_code",
        "reason_code_2",
        "review_category"
    ]
].copy()

reviewer_queue.head(20)

,rank,content_hash_id,score,impressions,current_ctr,avg_position,position_bucket,reason_code,reason_code_2,review_category
0,1,content_95dff98b3e9e533d,1.0,1476.0,0.124661,4.405827,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
1,2,content_fd99aed0910e521d,1.0,5409.0,0.105565,1.946755,Top3,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
2,3,content_421766adc84fcd7f,1.0,761.0,0.086728,4.643890,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
3,4,content_e31548c47c905839,1.0,1698.0,0.086572,5.262662,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
4,5,content_7811a2ed858d5816,1.0,606.0,0.080858,5.584158,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
5,6,content_dc5e5285b1691ea7,1.0,521.0,0.080614,7.886756,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
6,7,content_d46321b2dce9da21,1.0,756.0,0.068783,4.503968,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
7,8,content_33f041db9383a3af,1.0,607.0,0.067545,6.214168,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW
8,9,content_d15252a62541754e,1.0,1011.0,0.066271,10.229476,11-20,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_POSITION_11_20,PERFORMANCE_REVIEW
9,10,content_fa3db31b4aeeda73,1.0,637.0,0.065934,7.998430,4-10,MODEL_SIGNAL_CURRENT_CTR,MODEL_SIGNAL_AVG_POSITION,PERFORMANCE_REVIEW


## 5. Limitations

*What this work cannot claim.*


### Target definition

The prediction target is whether a page's CTR declines in the following month. This target was defined for this capstone analysis and is not a direct FlyRank-provided business outcome.

A CTR decline does not mean that a page will benefit from a refresh, expansion, protection, pruning, or monitoring action.

### Limited temporal evaluation

The temporal analysis uses five consecutive month-pairs from January–February through May–June 2026. The May–June 2026 pair was used as the held-out test period.

Therefore, the reported model performance represents one future-period evaluation and should not be treated as a universal estimate of future performance.

### Observational data

The analysis uses observed search-performance signals. It does not establish that any particular page characteristic causes future CTR decline.

The model's feature contributions and reason codes describe associations used for ranking, not causal explanations.

### Review action is not automated

The model produces a review-priority ranking. It does not automatically determine whether a page should be refreshed, expanded, protected, pruned, or monitored.

The final action requires human review and additional page-level context that is not available in the warehouse.

### Generalization

The model was evaluated on a single held-out future period. Performance may change on other months, clients, content types, or data distributions.

Additional temporal validation would be required before using the model as a production decision-support system.

### Data availability and privacy

The analysis intentionally uses observable search-performance signals and excludes private identifiers, URLs, private queries, credentials, future outcome features, and production action flags.

These restrictions improve the safety and reproducibility of the analysis but also limit the contextual information available to the model.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



The final reviewer queue ranks pages by the Logistic Regression score for the probability of CTR decline in the following month.

The ranking is intended to determine **which pages should be reviewed first**, not which action should automatically be taken.

### How reviewers should use the queue

Pages near the top of the queue should receive earlier human review because they have higher model scores on the held-out evaluation design.

For each page, the queue provides:

- `rank` — review priority based on the model score.
- `content_hash_id` — pseudonymous page identifier.
- `score` — model-estimated probability of the target CTR-decline outcome.
- `impressions` — current-month search visibility.
- `current_ctr` — current-month search click-through rate.
- `avg_position` — impression-weighted average search position.
- `position_bucket` — broad search-position context.
- `reason_code` — strongest positive model contribution associated with the score.
- `reason_code_2` — second positive model contribution when available.
- `review_category` — a reviewer-oriented starting category.

### Action framework

The model does not select the final SEO action. Reviewers can use the queue together with additional page-level context:

| Review signal | Possible review direction |
|---|---|
| Strong CTR-related signal with meaningful visibility | Review title, snippet, search intent alignment, and content relevance before considering a refresh or expansion |
| Poor visibility / position above 20 | Review discoverability, relevance, internal linking, and whether the page should continue receiving investment |
| Mid-ranking with weak engagement | Review whether the page sufficiently satisfies the likely search intent and whether improvement is justified |
| Strong performance but future decline risk | Review for protection, monitoring, or preventive maintenance rather than assuming a major content change is required |

These directions are review prompts, not automatic decisions.

### Recommended workflow

1. Start with the highest-ranked pages.
2. Inspect the model reason codes and current search signals.
3. Add page-level business and content context that is not present in the warehouse.
4. Decide whether the appropriate response is refresh, expansion, protection, pruning, monitoring, or no action.
5. Record the decision and its eventual outcome.
6. Use those independently recorded outcomes for future evaluation of the prioritization system.

This workflow keeps the model responsible for prioritization while keeping the final content decision with a human reviewer.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
